# Глава 19. AutoML: ускоряем работу

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь объяснить, что автоматизирует AutoML, запустить AutoML на табличной задаче и сравнить с ручной моделью, а также взять готовую модель с Hugging Face Hub.

Во второй фазе ты вручную собирал Pipeline, выбирал модель, подбирал параметры. Это важнейший навык — без него ты не поймёшь, что происходит. Но что, если нужно получить хороший результат быстро, а времени в обрез? На помощь приходит AutoML (Automated Machine Learning): инструменты, которые сами перебирают модели и настройки за тебя.

**Аналогия: автопилот в машине  💡**

AutoML — это автопилот. Ты по-прежнему должен знать, куда едешь и как в целом устроена машина (иначе не заметишь, если автопилот везёт не туда, и не сможешь вмешаться). Но рутину — держать полосу, подстраивать скорость — берёт на себя система. AutoML сам пробует десятки моделей и комбинаций параметров и выдаёт лучшую. Твоя задача — грамотно поставить цель, подготовить данные и проверить результат. Автопилот не отменяет водителя.

## Что AutoML делает за тебя

AutoML автоматизирует самую трудоёмкую и рутинную часть работы. Он сам пробует разные алгоритмы (лес, градиентный бустинг, KNN и другие), сам подбирает их параметры, сам сравнивает результаты по кросс-валидации и выбирает победителя. То, на что вручную уходят часы перебора, он делает за минуты.

Этап

Вручную (фаза 2)

С AutoML

Выбор модели

Пробуешь по одной сам

Перебирает десятки автоматически

Настройка параметров

GridSearchCV вручную

Подбирает сам

Сравнение моделей

Считаешь и сравниваешь сам

Строит готовый рейтинг

## AutoML в коде

Популярные инструменты — AutoGluon и H2O. Возьмём AutoGluon: он решает табличную задачу буквально в несколько строк, а под капотом делает всё то, чему мы учились целую фазу.

In [ ]:
# Запускать в Google Colab (нужен интернет для установки)
!pip install autogluon.tabular -q

import pandas as pd
from autogluon.tabular import TabularPredictor

# Данные о пингвинах — та же задача, что решали вручную в фазе 2
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(url).drop(columns=["sex"])
train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index)

# AutoML: одна команда — и он сам перебирает модели
predictor = TabularPredictor(label="species").fit(train)

# Рейтинг всех опробованных моделей
predictor.leaderboard(test)

_Ожидаемый результат: таблица-рейтинг моделей, отсортированная по точности на тесте._

Посмотри на leaderboard — AutoGluon показывает десятки моделей, которые он попробовал, с их точностью. Ты за одну команду получил то, на что в фазе 2 ушло бы много ручной работы.

## Главное сравнение: вручную против AutoML

Возьми точность своей ручной модели из фазы 2 на пингвинах и сравни с лучшим результатом AutoML на тех же данных. Обычно AutoML немного выигрывает — но не в разы, а на считанные проценты. И это важнейший вывод главы, а не разочарование.

**Внимание: AutoML — не волшебство  ⚠️**

AutoML не заменяет понимание. Он не поставит задачу за тебя, не заметит утечку данных, не выберет правильную метрику под задачу и не объяснит, почему модель ошибается. Если данные плохие или есть утечка — AutoML выдаст плохой (или обманчиво хороший) результат быстро и уверенно. Инструмент экономит время эксперту, но не превращает новичка в эксперта. Именно поэтому мы сначала прошли фазу 2 руками — чтобы уметь проверить и понять то, что делает автопилот.

## Hugging Face Hub: библиотека готовых моделей

**Аналогия: магазин приложений для ИИ  💡**

Hugging Face Hub — это как магазин приложений, только для моделей машинного обучения. Тысячи готовых, уже обученных моделей: перевод, распознавание речи, анализ тональности, генерация картинок, ответы на вопросы. Многие можно запустить в пару строк, не обучая вообще ничего. Прежде чем строить модель самому, загляни туда — очень возможно, что кто-то уже сделал именно то, что тебе нужно, и выложил бесплатно.

In [ ]:
# Запускать в Google Colab
!pip install transformers -q
from transformers import pipeline

# Берём готовую модель анализа тональности одной строкой
classifier = pipeline("sentiment-analysis")

print(classifier("I love this course!"))
print(classifier("This is terrible."))

_Ожидаемый результат: метка POSITIVE/NEGATIVE и уверенность для каждой фразы._

Слово pipeline здесь — из библиотеки Hugging Face, и это НЕ тот же самый sklearn Pipeline из фазы 2 (одинаковое слово, разные вещи — частая путаница). Здесь pipeline скачивает готовую модель и сразу применяет её к тексту. Никакого обучения — модель уже умеет.

## Мини-AutoML своими руками

Чтобы прочувствовать, что AutoML не магия, можно собрать его крошечную версию самому — просто перебрать несколько моделей в цикле и сравнить по кросс-валидации. Это ровно то, что делает AutoGluon, только у него моделей и настроек больше.

In [ ]:
# === Данные для мини-рейтинга (выполни, если открыл новый блокнот) ===
import pandas as pd
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(url).dropna(subset=["bill_length_mm", "body_mass_g", "species"])
X = df[["bill_length_mm", "body_mass_g"]]
y = df["species"]

from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

candidates = {
    "KNN": KNeighborsClassifier(),
    "Дерево": DecisionTreeClassifier(random_state=42),
    "Лес": RandomForestClassifier(random_state=42),
}
# Перебираем модели и строим свой мини-рейтинг
for name, model in candidates.items():
    score = cross_val_score(model, X, y, cv=5).mean()
    print(f"{name:7}: {score:.3f}")

## Задания

Базовый уровень

Запусти AutoGluon на пингвинах и посмотри leaderboard. Сравни лучший результат с точностью своей ручной модели из фазы 2. Насколько AutoML лучше — на проценты или в разы?

Со звёздочкой  ⭐

Возьми готовую модель перевода или ответов на вопросы с Hugging Face (transformers pipeline) и примени к своему тексту. Затем реши: для какой задачи выгоднее взять готовую модель, а для какой — обучать свою? Обоснуй.

Вопросы для обсуждения  💬

Если AutoML почти всегда не хуже ручной работы, зачем вообще учиться делать модели руками?

Что AutoML НЕ сделает за тебя? Перечисли хотя бы три вещи.

В каких случаях лучше взять готовую модель с Hugging Face, а не обучать свою?

**Проверь себя  ℹ️**

1. Что именно автоматизирует AutoML?

2. Почему AutoML не заменяет знание основ ML?

3. Чем pipeline из Hugging Face отличается от Pipeline из sklearn?